<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Bioinformatics_DCA/blob/main/5_clase_alineamientos_de_secuencias_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Alineamientos de secuencias por pares (Pairwise) en Biopython

**Nota para Google Colab:**  
- Ejecuta `!pip install biopython` si aún no lo tienes instalado.  
- Este notebook usa el módulo `Bio.pairwise2` (aunque está marcado como deprecado, sigue funcionando en las versiones actuales de Biopython).

**Alineamiento de secuencias por pares** se utiliza para identificar regiones de similitud que pueden indicar relaciones funcionales, estructurales y/o evolutivas entre dos secuencias biológicas (proteínas o ácidos nucleicos).

Identificar la región similar nos permite inferir mucha información, como qué rasgos se conservan entre especies, qué tan cercanas genéticamente están diferentes especies, cómo evolucionan las especies, etc.

El alineamiento de secuencias por pares utiliza programación dinámica para encontrar el alineamiento óptimo entre las dos secuencias, puntuando según su similitud (qué tan similares son) o distancia (qué tan diferentes son), y luego evaluando la significancia de esta puntuación.

### Tipos de alineamientos por pares

1. **Alineamiento global**: Este método encuentra el mejor alineamiento a lo largo de toda la longitud de las 2 secuencias. ¿Cuál es la máxima similitud entre la secuencia X y la Y?

2. **Alineamiento local**: Este método encuentra las subsecuencias más similares entre las 2 secuencias. ¿Cuál es la máxima similitud entre una subsecuencia de X y una subsecuencia de Y?

Al realizar alineamientos, puedes especificar la puntuación de coincidencia (*match score*) y las penalizaciones por huecos (*gap penalties*).

1. La **puntuación de coincidencia (match score)** indica la compatibilidad entre el alineamiento de dos caracteres en las secuencias. Los caracteres altamente compatibles deben recibir puntuaciones positivas, y los incompatibles deben recibir puntuaciones negativas o 0.

2. Las **penalizaciones por huecos (gap penalties)** deben ser negativas.

### Bio.pairwise2

Biopython incluye dos alineadores por pares integrados: el módulo `Bio.pairwise2` y la clase `PairwiseAligner` dentro del módulo `Bio.Align` (desde la versión 1.72 de Biopython). Ambos pueden realizar alineamientos globales y locales. → Nos enfocaremos en **pairwise2**.

Los nombres de las funciones de alineamiento en este módulo siguen la convención **alignmenttypeXY**, donde **alignmenttype** es “global” o “local” y **XY** es un código de 2 caracteres que indica los parámetros que toma. El primer carácter **X** indica los parámetros para coincidencias (y no coincidencias), y el segundo **Y** indica los parámetros para las penalizaciones por huecos.

Los parámetros de coincidencia (*match*) son:

1. **x**  –   Sin parámetros. Los caracteres idénticos tienen puntuación de 1, de lo contrario 0.

2. **m**  –   Una puntuación de coincidencia es la puntuación de caracteres idénticos; de lo contrario, puntuación de no coincidencia. Palabras clave: **match**, **mismatch**.

3. **d**  –   Un diccionario devuelve la puntuación de cualquier par de caracteres. Palabra clave: **match_dict**.

4. **c**  –   Una función de callback devuelve las puntuaciones. Palabra clave: **match_fn**.

Los parámetros de penalización por huecos (*gap*) son:

1. **x**  –   Sin penalizaciones por huecos.

2. **s**  –   Mismas penalizaciones de apertura y extensión de huecos para ambas secuencias. Palabras clave: **open**, **extend**.

3. **d**  –   Las secuencias tienen diferentes penalizaciones de apertura y extensión de huecos. Palabras clave: **openA**, **extendA**, **openB**, **extendB**.

4. **c**  –   Una función de callback devuelve las penalizaciones por huecos. Palabras clave: **gap_A_fn**, **gap_B_fn**.

### Ejemplos de alineamiento global

Para el alineamiento local usamos las mismas funciones, ¡solo que en lugar de llamar a *global*, llamamos a *local*!

In [ ]:
from Bio import pairwise2

In [ ]:
# globalxx - coincidencias puntúan 1, no coincidencias 0 y sin penalización por huecos.
alignments = pairwise2.align.globalxx(
    "ACGTACGTCGATGATCGTACGTACGTACGTCGTAGTGATCGGCTGATCATCGTAGCATCGATCGTACTACGT",
    "AGCGACGATCGACTGCTGATCACGT"
)
for alignment in alignments:
    print(pairwise2.format_alignment(*alignment))

In [ ]:
# globalmx - coincidencias puntúan 2, no coincidencias -1. Sin penalización por huecos.
alignments = pairwise2.align.globalmx(
    "ACCGGTACCGGTACCGGTACCGGTACCGGTACCGGTACCGGTACCGGT",
    "ACCGGTACCGGTACCGGTACCGGTACCGGTACGT",
    match=2,
    mismatch=-1
)
for alignment in alignments:
    print(pairwise2.format_alignment(*alignment))

In [ ]:
# globalxs - coincidencias puntúan 1, no coincidencias 0, apertura de hueco -2, extensión de hueco -1
alignments = pairwise2.align.globalxs(
    "ACCGGTACCGGTACCGGTACCGGT",
    "ACCGGTACCGGTACCGGTACCGGT",
    open=-2,
    extend=-1
)
for alignment in alignments:
    print(pairwise2.format_alignment(*alignment))

In [ ]:
# globaldx - puntuaciones de coincidencia/no coincidencia leídas de la matriz BLOSUM62, sin penalización por huecos
from Bio.Align import substitution_matrices

matrix = substitution_matrices.load("BLOSUM62")  # matriz de puntuación BLOSUM62 para alineamiento de proteínas
alignments = pairwise2.align.globaldx(
    "KEVLAKEVLAKEVLAKEVLAKEVLA",
    "EKEVLAKEVLAKEVLAKEVLAVL",
    match_dict=matrix
)
for alignment in alignments:
    print(pairwise2.format_alignment(*alignment))

In [ ]:
# globalmc - coincidencias puntúan 5, no coincidencias -4, penalización por huecos definida mediante la función gap_function
from math import log

def gap_function(x, y):  # x es la posición del hueco en la secuencia, y es la longitud del hueco
    if y == 0:  # Sin hueco
        return 0
    elif y == 1:  # Penalización por apertura de hueco
        return -2
    return -(2 + y / 4.0 + log(y) / 2.0)

alignments = pairwise2.align.globalmc(
    "ACCCCCGTACCCCCGTACCCCCGTACCCCCGT",
    "AACCCCCGTACCCCCGTACCCCCGTCG",
    match=5,
    mismatch=-4,
    gap_A_fn=gap_function,
    gap_B_fn=gap_function
)
for alignment in alignments:
    print(pairwise2.format_alignment(*alignment))